In [ ]:
!pip install torch torchvision torchaudio
!pip install pandas numpy matplotlib tqdm opencv-python

!pip install -q opencv-python==4.10.0.84
!pip install -q opencv-contrib-python==4.10.0.84
!pip install -q ultralytics
!pip install -q numpy==1.23.5 --force-reinstall
!pip install -q matplotlib==3.9.0
!pip install -q tqdm==4.66.4

print("\n✓ パッケージのインストールが完了しました")

In [ ]:
# Google Driveをマウント（Colab環境の場合）
import sys
import os
from pathlib import Path

try:
    import google.colab
    from google.colab import drive
    
    # Google Driveをマウント
    drive.mount('/content/drive')
    
    # プロジェクトディレクトリに移動
    PROJECT_ROOT = '/content/drive/MyDrive/Visuable_for_you_tabletennis'
    os.chdir(PROJECT_ROOT)
    
    # notebooksディレクトリをパスに追加
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'scripts/notebooks'))
    
    print(f"✓ Google Driveをマウントしました")
    print(f"✓ プロジェクトルート: {PROJECT_ROOT}")
    IN_COLAB = True
    
except ImportError:
    # ローカル環境の場合
    IN_COLAB = False
    # notebooksディレクトリ（このノートブックの場所）をパスに追加
    notebook_dir = Path(__file__).parent if '__file__' in globals() else Path.cwd()
    sys.path.insert(0, str(notebook_dir))
    print(f"✓ ローカル環境で実行中")
    print(f"✓ 作業ディレクトリ: {Path.cwd()}")

# utilsをインポート
from utils import ColabFileManager, ConfigLoader

# ファイルマネージャーの初期化（プロジェクトルートは自動検出）
fm = ColabFileManager()

print(f"✓ ファイルマネージャー初期化完了")
print(f"  - 検出されたプロジェクトルート: {fm.project_root}")
print(f"  - Colab環境: {fm.is_colab}")

In [ ]:
import torch
import json
from pathlib import Path

# パイプラインのインポート
from src.pipelines import InferencePipeline
from src.pipelines.config import InferencePipelineConfig

print("✓ モジュールのインポートが完了しました")
print(f"PyTorchバージョン: {torch.__version__}")
print(f"CUDAが利用可能: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 設定ファイルのパス
CONFIG_PATH = 'configs/crip_app_config.json'

# 設定の読み込み
if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH, 'r') as f:
        config_dict = json.load(f)
    print(f"✓ 設定ファイルを読み込みました: {CONFIG_PATH}")
else:
    print(f"警告: 設定ファイルが見つかりません: {CONFIG_PATH}")

# 設定の表示
print("\n設定内容:")
for key, value in config_dict.items():
    print(f"  {key}: {value}")

In [ ]:
# ============================================================
# パス設定（ここを変更してください）
# ============================================================

# 入力動画
INPUT_VIDEO = 'data/raw_01_test/sample_video_03_short.mp4'
path = Path(INPUT_VIDEO).stem

# 出力ディレクトリ
OUTPUT_DIR = f'output/predictions/{path}'

# ベース名（Noneの場合は入力動画名を使用）
BASE_NAME = None

# ============================================================

# ファイル存在確認
print("="*60)
print("ファイルの確認")
print("="*60)
print(f"入力動画: {'✓' if os.path.exists(INPUT_VIDEO) else '✗'} {INPUT_VIDEO}")
print(f"卓球台検出モデル: {'✓' if os.path.exists(config_dict['models']['table_detection']) else '✗'} {config_dict['models']['table_detection']}")
print(f"姿勢推定モデル: {'✓' if os.path.exists(config_dict['models']['pose_estimation']) else '✗'} {config_dict['models']['pose_estimation']}")
print(f"プレー分類モデル: {'✓' if os.path.exists(config_dict['models']['play_classifier']) else '✗'} {config_dict['models']['play_classifier']}")
print(f"出力ディレクトリ: {OUTPUT_DIR}")
print("="*60)

In [ ]:
# InferencePipelineConfigを作成
from src.pipelines.config import (
    PlayerPoseExporterConfig,
    TableDetectionConfig,
    PoseTrackingConfig,
    PlayerClassificationConfig,
    TrackingExportConfig,
    VideoProcessingConfig,
    PlaySceneDetectionConfig,
)

# 各設定を作成
table_detection_config = TableDetectionConfig(
    model_path=config_dict['models']['table_detection'],
    device=config_dict.get('device', 'cuda'),
    min_confidence=config_dict.get('table_detection', {}).get('min_confidence', 0.5),
    max_detection_attempts=config_dict.get('table_detection', {}).get('max_detection_attempts', 200)
)

pose_tracking_config = PoseTrackingConfig(
    model_path=config_dict['models']['pose_estimation'],
    device=config_dict.get('device', 'cuda'),
    conf_threshold=config_dict.get('pose_tracking', {}).get('conf_threshold', 0.5),
    iou_threshold=config_dict.get('pose_tracking', {}).get('iou_threshold', 0.7),
    table_distance_threshold=config_dict.get('pose_tracking', {}).get('table_distance_threshold', 0.2),
    min_keypoint_confidence=config_dict.get('pose_tracking', {}).get('min_keypoint_confidence', 0.5),
    half=config_dict.get('pose_tracking', {}).get('half', False)
)

player_classification_config = PlayerClassificationConfig(
    near_table_threshold=config_dict.get('player_classification', {}).get('near_table_threshold', 0.1),
    min_tracking_frames=config_dict.get('player_classification', {}).get('min_tracking_frames', 10),
    max_players=config_dict.get('player_classification', {}).get('max_players', 4),
    max_inactive_frames=config_dict.get('player_classification', {}).get('max_inactive_frames', 30),
    min_player_score=config_dict.get('player_classification', {}).get('min_player_score', 0.3),
    recent_frames_window=config_dict.get('player_classification', {}).get('recent_frames_window', 146),
    max_consecutive_other_count=config_dict.get('player_classification', {}).get('max_consecutive_other_count', 30),
    movement_noise_threshold=config_dict.get('player_classification', {}).get('movement_noise_threshold', 5.0)
)

tracking_export_config = TrackingExportConfig(
    min_consecutive_frames=config_dict.get('tracking_export', {}).get('min_consecutive_frames', 30),
    max_frame_gap=config_dict.get('tracking_export', {}).get('max_frame_gap', 5)
)

video_processing_config = VideoProcessingConfig(
    target_fps=config_dict.get('video_processing', {}).get('target_fps', 30.0),
    show_progress=config_dict.get('video_processing', {}).get('show_progress', True),
    output_codec=config_dict.get('video_processing', {}).get('output_codec', 'mp4v')
)

pose_export_config = PlayerPoseExporterConfig(
    table_detection=table_detection_config,
    pose_tracking=pose_tracking_config,
    player_classification=player_classification_config,
    tracking_export=tracking_export_config,
    video_processing=video_processing_config,
    save_intermediate_files=config_dict.get('pipeline', {}).get('save_intermediate_files', True)
)

scene_detection_config = PlaySceneDetectionConfig(
    model_path=config_dict['models']['play_classifier'],
    config_path=config_dict['models'].get('play_classifier_config'),
    device=config_dict.get('device', 'cuda'),
    threshold=config_dict.get('scene_detection', {}).get('threshold', 0.5),
    min_scene_duration=config_dict.get('scene_detection', {}).get('min_scene_duration', 10)
)

pipeline_config = InferencePipelineConfig(
    pose_export=pose_export_config,
    scene_detection=scene_detection_config,
    show_progress=config_dict.get('pipeline', {}).get('show_progress', True),
    save_intermediate_files=config_dict.get('pipeline', {}).get('save_intermediate_files', True)
)

# パイプラインの初期化
pipeline = InferencePipeline(config=pipeline_config)

print("\n✓ パイプライン初期化完了")

In [ ]:
# パイプライン実行
results = pipeline.process_video(
    input_video=INPUT_VIDEO,
    output_dir=OUTPUT_DIR,
    base_name=BASE_NAME
)

print("\n✓ パイプライン処理が完了しました")

In [ ]:
import cv2
from tqdm import tqdm

def clip_play_scenes(
    input_video: str,
    scenes: list,
    output_dir: str,
    base_name: str,
    fps: float,
    buffer_before_sec: float = 0.5,
    buffer_after_sec: float = 0.5,
):
    """
    検出されたプレーシーンを個別の動画ファイルとして切り出す

    Args:
        input_video: 入力動画パス
        scenes: [(start_frame, end_frame), ...] のリスト
        output_dir: 出力ディレクトリ
        base_name: 出力ファイルのベース名
        fps: 動画のFPS
        buffer_before_sec: シーン開始前のバッファ（秒）
        buffer_after_sec: シーン終了後のバッファ（秒）

    Returns:
        出力ファイルパスのリスト
    """
    if not scenes:
        print("切り出すシーンがありません")
        return []

    cap = cv2.VideoCapture(input_video)
    if not cap.isOpened():
        raise RuntimeError(f"動画を開けません: {input_video}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    src_fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # FPSの比率（パイプラインがtarget_fpsにリサンプルしている場合の補正）
    fps_ratio = src_fps / fps if fps > 0 else 1.0

    buffer_before = int(buffer_before_sec * fps)
    buffer_after = int(buffer_after_sec * fps)

    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    output_paths = []
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')

    print(f"\nPlay Scene Clipping")
    print(f"{'='*60}")
    print(f"  Input: {input_video}")
    print(f"  Source FPS: {src_fps:.1f}, Pipeline FPS: {fps:.1f}")
    print(f"  Resolution: {width}x{height}")
    print(f"  Scenes to clip: {len(scenes)}")
    print(f"  Buffer: -{buffer_before_sec}s / +{buffer_after_sec}s")
    print(f"{'='*60}")

    for i, (start_frame, end_frame) in enumerate(scenes, 1):
        # バッファ付きのフレーム範囲（パイプラインFPS基準）
        clip_start = max(0, int(start_frame) - buffer_before)
        clip_end = min(total_frames - 1, int(end_frame) + buffer_after)

        # 元動画のフレーム番号に変換
        src_start = int(clip_start * fps_ratio)
        src_end = int(clip_end * fps_ratio)
        src_end = min(src_end, total_frames - 1)

        duration_frames = src_end - src_start + 1
        duration_sec = duration_frames / src_fps

        out_path = out_dir / f"{base_name}_scene_{i:03d}.mp4"
        writer = cv2.VideoWriter(str(out_path), fourcc, src_fps, (width, height))

        cap.set(cv2.CAP_PROP_POS_FRAMES, src_start)

        for _ in tqdm(range(duration_frames), desc=f"  Scene {i}/{len(scenes)}", leave=False):
            ret, frame = cap.read()
            if not ret:
                break
            writer.write(frame)

        writer.release()
        output_paths.append(str(out_path))
        print(f"  Scene {i}: frame {start_frame}-{end_frame} -> {out_path.name} ({duration_sec:.1f}s)")

    cap.release()

    # 全シーンを結合したハイライト動画も作成
    if len(scenes) > 0:
        highlight_path = out_dir / f"{base_name}_highlights.mp4"
        highlight_writer = cv2.VideoWriter(str(highlight_path), fourcc, src_fps, (width, height))

        cap = cv2.VideoCapture(input_video)
        for i, (start_frame, end_frame) in enumerate(scenes, 1):
            clip_start = max(0, int(start_frame) - buffer_before)
            clip_end = min(total_frames - 1, int(end_frame) + buffer_after)
            src_start = int(clip_start * fps_ratio)
            src_end = int(clip_end * fps_ratio)
            src_end = min(src_end, total_frames - 1)

            cap.set(cv2.CAP_PROP_POS_FRAMES, src_start)
            for _ in range(src_end - src_start + 1):
                ret, frame = cap.read()
                if not ret:
                    break
                highlight_writer.write(frame)

        highlight_writer.release()
        cap.release()

        total_highlight_sec = os.path.getsize(str(highlight_path)) / 1024 / 1024
        print(f"\n  Highlights: {highlight_path.name} ({total_highlight_sec:.1f} MB)")
        output_paths.append(str(highlight_path))

    print(f"\nClipping complete: {len(scenes)} scenes extracted")
    return output_paths


# プレーシーンの切り出し実行
target_fps = config_dict.get('video_processing', {}).get('target_fps', 30.0)

clip_paths = clip_play_scenes(
    input_video=INPUT_VIDEO,
    scenes=results['scene_detection']['scenes'],
    output_dir=OUTPUT_DIR,
    base_name=results.get('input_video', INPUT_VIDEO).replace('/', '_').replace('.', '_')
        if not BASE_NAME else BASE_NAME,
    fps=target_fps,
    buffer_before_sec=0.5,
    buffer_after_sec=0.5,
)

results['output_files']['clip_videos'] = clip_paths
if clip_paths:
    results['output_files']['highlights_video'] = clip_paths[-1]

In [ ]:
from IPython.display import Video, display, Image
import matplotlib.pyplot as plt

print("="*70)
print("Processing Summary")
print("="*70)
print(f"\nInput Video: {results['input_video']}")
print(f"Output Dir:  {results['output_dir']}")

print(f"\n[Pose Extraction]")
print(f"  Processed Frames: {results['pose_export']['processed_frames']}")
print(f"  Detected Players: {results['pose_export']['player_ids']}")
print(f"  Pose CSV:   {results['output_files'].get('pose_csv', 'N/A')}")
print(f"  Pose Video: {results['output_files'].get('pose_video', 'N/A')}")

print(f"\n[Play Scene Detection]")
print(f"  Detected Scenes: {results['scene_detection']['total_scenes']}")
print(f"  Threshold: {results['scene_detection']['threshold']}")
print(f"  Min Scene Duration: {results['scene_detection']['min_scene_duration']} frames")

if results['scene_detection']['total_scenes'] > 0:
    print(f"\n  Scenes (first 5):")
    for i, (start, end) in enumerate(results['scene_detection']['scenes'][:5], 1):
        duration = end - start + 1
        print(f"    Scene {i}: frame {start}-{end} ({duration} frames)")

print(f"\n[Clipped Videos]")
clip_videos = results['output_files'].get('clip_videos', [])
highlights = results['output_files'].get('highlights_video')
if clip_videos:
    for p in clip_videos:
        name = Path(p).name
        size_mb = os.path.getsize(p) / 1024 / 1024 if os.path.exists(p) else 0
        print(f"  {name} ({size_mb:.1f} MB)")
else:
    print("  No clips generated")

print("="*70)

# Prediction graph
output_base = Path(results['output_dir']) / Path(results['input_video']).stem
graph_path = f"{output_base}_prediction_graph.png"

if os.path.exists(graph_path):
    print(f"\nPrediction Graph:")
    display(Image(graph_path))

# Highlight video preview
if highlights and os.path.exists(highlights):
    print(f"\nHighlights Video Preview:")
    display(Video(highlights, width=800))